# 05 · Previsões com o modelo vigente

**Feature F8** (critério 8) · dono: ver `config/equipe.yaml` · funções em `src/promocao.py`

| | |
|---|---|
| **Entradas** | `models/modelo_vigente.json` e o arquivo do vigente em `models/registro/`, o próximo lote de `data/incoming/` |
| **Saídas** | `evidencias/previsoes/previsoes_<cycle_id>.csv` · conferência das previsões de ciclos anteriores |

**Regras**
- O vigente é o `Pipeline` completo da F5: pré-processamento, seleção das features eleitas e algoritmo.
- As previsões são feitas no **próximo lote de `data/incoming/`**: reservas que ainda não entraram no pipeline e não participaram da escolha do modelo (o teste do ciclo já foi usado na promoção, no notebook 04).
- O lote passa pelo mesmo ETL do notebook 01. O lote simulado traz o rótulo, porque o dataset é histórico, mas ele não é usado na previsão nem gravado com ela.
- Quando o lote chega no ciclo seguinte, as previsões são conferidas com o rótulo real.
- Sem vigente ou sem lote restante, o notebook só avisa e termina sem erro.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto, resolver
from src.preprocessamento import carregar_particao
from src import promocao

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

cycle_id=20261120T012057Z-local | data_version=dv018-f95c3940


## 1. Modelo vigente

In [2]:
manifesto = promocao.carregar_manifesto(cfg, RAIZ)
modelo, vigente = promocao.carregar_vigente(manifesto, RAIZ)
if vigente is None:
    print("AVISO: não há modelo vigente (nenhum ciclo promoveu ainda). Nada a prever.")
else:
    print(f"vigente: modelo {vigente['modelo']} ({vigente['algoritmo']}) do ciclo {vigente['cycle_id']}")
    print(f"data_version do treino: {vigente['data_version']} | {len(vigente['features'])} features | arquivo: {vigente['arquivo']}")

vigente: modelo B (random_forest) do ciclo 20261120T012057Z-local
data_version do treino: dv018-f95c3940 | 17 features | arquivo: modelo_20261120T012057Z-local_B.joblib


## 2. Previsões no próximo lote

In [3]:
arquivo_lote = promocao.proximo_lote(cfg, RAIZ)
previsoes = None
if vigente is None:
    pass
elif arquivo_lote is None:
    print("AVISO: não há mais lotes em data/incoming. Sem lote futuro para prever neste ciclo.")
else:
    lote, rel = promocao.preparar_lote(arquivo_lote, cfg)
    previsoes = promocao.prever(modelo, lote, cfg)
    caminho = promocao.salvar_previsoes(previsoes, ctx, vigente, RAIZ)
    tempo = cfg["avaliacao"]["coluna_tempo"]
    print(f"lote: {arquivo_lote.name} | {rel['n_entrada']:,} linhas brutas, {rel['removidas_total']:,} removidas pelo ETL, {len(previsoes):,} previstas")
    print(f"chegadas de {previsoes[tempo].min():%Y-%m-%d} a {previsoes[tempo].max():%Y-%m-%d}")
    print(f"cancelamento previsto: {previsoes['previsao'].mean():.1%} | probabilidade média: {previsoes['prob_cancelamento'].mean():.3f}")
    print("salvo em", caminho.relative_to(RAIZ))
    display(previsoes.head())

lote: lote_2017-01.csv.gz | 3,681 linhas brutas, 8 removidas pelo ETL, 3,673 previstas
chegadas de 2017-01-01 a 2017-01-31
cancelamento previsto: 22.3% | probabilidade média: 0.365
salvo em evidencias\previsoes\previsoes_20261120T012057Z-local.csv


,lote_id,linha_origem,arrival_date,prob_cancelamento,previsao
0,lote_2017-01,0,2017-01-01,0.345171,0
1,lote_2017-01,1,2017-01-01,0.404815,0
2,lote_2017-01,2,2017-01-01,0.404815,0
3,lote_2017-01,3,2017-01-01,0.404815,0
4,lote_2017-01,4,2017-01-01,0.374439,0


## 3. Conferência das previsões anteriores

Previsões de ciclos anteriores cujo lote já entrou no pipeline, comparadas com o rótulo real.

In [4]:
conferencia = promocao.conferir_previsoes_anteriores(cfg, RAIZ)
if conferencia.empty:
    print("Nenhuma previsão anterior com lote já incorporado para conferir.")
else:
    display(conferencia.round(4))

Nenhuma previsão anterior com lote já incorporado para conferir.


## 4. O que mudou entre os ciclos

In [5]:
historico = promocao.tabela_historico(manifesto)
if historico.empty:
    print("Nenhuma decisão registrada ainda.")
else:
    display(historico.round(4))

,cycle_id,data_version,vencedor,algoritmo,valor_vencedor,vigente_anterior,valor_vigente,decisao
0,20261120T012057Z-local,dv018-f95c3940,B,random_forest,0.6773,None,None,promovido
